In [1]:
# ============================================================
# Import Required Libraries
# ============================================================

import os
import re
import json
import torch
import torch.nn.functional as F

from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

print("=" * 60)
print("Libraries Imported Successfully")
print("=" * 60)

Libraries Imported Successfully


In [2]:
# ============================================================
# GPU Configuration
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 60)
print("Device :", device)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

print("=" * 60)

Device : cuda
GPU : NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [3]:
# ============================================================
# Load Teacher Model
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True

)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

teacher_model = AutoModelForCausalLM.from_pretrained(

    MODEL_NAME,

    quantization_config=bnb_config,

    torch_dtype=torch.float16,

    device_map="auto"

)

teacher_model.eval()

print("=" * 60)
print("Teacher Model Loaded Successfully")
print("=" * 60)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
W0728 16:51:15.312000 25800 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Teacher Model Loaded Successfully


In [23]:
# ============================================================
# Teacher Evaluation Prompt
# ============================================================

teacher_prompt = """
You are a senior board-certified thoracic radiologist with over 25 years of experience.

Your ONLY task is to evaluate the quality of a student-generated radiology report by comparing it with the Ground Truth Report.

You are NOT generating a radiology report.

You are ONLY grading the student report.

===========================================================
EVALUATION GUIDELINES
===========================================================

Compare the Student Report with the Ground Truth Report.

Evaluate based on:

• Correct medical findings
• Missed important findings
• Hallucinated findings
• Clinical reasoning
• Final diagnosis

Ignore wording differences.

Ignore report style.

Ignore report length.

Focus ONLY on factual medical correctness.

Be strict and objective.

===========================================================
SCORING
===========================================================

Assign ONE final quality score between 0.0 and 10.0.

Scoring Guide:

10 = Nearly perfect report

8–9 = Very good with minor mistakes

6–7 = Mostly correct but several mistakes

4–5 = Moderate quality with important errors

2–3 = Poor report with many errors

0–1 = Completely incorrect or hallucinated report

You may use increments of 0.5 only.

===========================================================
OUTPUT FORMAT
===========================================================

Return ONLY ONE line.

Score: X.X/10

Do NOT explain.

Do NOT justify.

Do NOT summarize.

Do NOT output anything else.
"""

In [24]:
# ============================================================
# Teacher Evaluation Function
# ============================================================

import re

def evaluate_trajectory(ground_truth, trajectory):

    prompt = f"""
{teacher_prompt}

===========================================================

GROUND TRUTH REPORT

{ground_truth}

===========================================================

STUDENT REPORT

{trajectory}

===========================================================

Evaluate the student report.
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(device)

    with torch.no_grad():

        outputs = teacher_model.generate(

            **inputs,

            max_new_tokens=40,

            do_sample=False,

            temperature=0.0,

            repetition_penalty=1.1,

            pad_token_id=tokenizer.eos_token_id,

            eos_token_id=tokenizer.eos_token_id

        )

    generated = outputs[0][inputs.input_ids.shape[1]:]

    response = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    # Extract only the score
    match = re.search(
        r"Score:\s*([0-9]+(?:\.[0-9]+)?)/10",
        response,
        re.IGNORECASE
    )

    if match:
        return float(match.group(1))

    return None

In [25]:
# ============================================================
# Load Student Trajectories
# ============================================================

import os
import json

TRAJECTORY_DIR = "generated_trajectories"

trajectory_files = sorted(
    [
        f for f in os.listdir(TRAJECTORY_DIR)
        if f.endswith(".json")
    ]
)

print(f"Found {len(trajectory_files)} trajectory files.")

Found 1 trajectory files.


In [26]:
# ============================================================
# Load One Sample
# ============================================================

sample_file = trajectory_files[0]

sample_path = os.path.join(
    TRAJECTORY_DIR,
    sample_file
)

with open(sample_path, "r") as f:
    sample = json.load(f)

ground_truth = sample["ground_truth"]

trajectories = sample["trajectories"]

print("Sample:", sample_file)
print("Ground Truth Loaded")
print("Number of Trajectories:", len(trajectories))

Sample: sample_000001.json
Ground Truth Loaded
Number of Trajectories: 5


In [27]:
# ============================================================
# Preview Data
# ============================================================

print("=" * 80)
print("GROUND TRUTH")
print("=" * 80)
print(ground_truth)

print("\n")

for i, traj in enumerate(trajectories):

    print("=" * 80)
    print(f"Trajectory {i+1}")
    print("=" * 80)
    print(traj)
    print()

GROUND TRUTH
AP portable upright view of the chest. Lung volumes are low limiting assessment. Mild bibasilar opacity is likely related to atelectasis. No large effusion or pneumothorax. Heart size appears normal. Mediastinal contour difficult to assess given patient rotation to the right. Chronic deformity at bilateral shoulders.  Bibasal atelectasis.


Trajectory 1
### Findings:

- **Heart Size and Mediastinum:** Normal cardiac silhouette without evidence of cardiomegaly.
- **Lungs:** Bilateral lung fields show diffuse, fine reticular opacities consistent with interstitial lung disease (ILD).
- **Pleural Space:** Absent findings of pleural effusion or pneumothorax.
- **Bones and Visible Soft Tissues:** Normal bony structures and visible soft tissues without fractures or significant abnormalities.

### Reasoning:

- **Lung Opacity:** The fine reticular opacities in both lung fields suggest interstitial lung disease (ILD). This is consistent with idiopathic pulmonary fibrosis or other c

In [28]:
# ============================================================
# Evaluate All Student Trajectories
# ============================================================

teacher_outputs = []

print("=" * 60)
print("Teacher Evaluation Started")
print("=" * 60)

for i, trajectory in enumerate(trajectories):

    print(f"\nEvaluating Trajectory {i+1}...")

    result = evaluate_trajectory(
        ground_truth,
        trajectory
    )

    teacher_outputs.append(result)

    print(f"Score: {result}/10")

print("\n" + "=" * 60)
print("Teacher Evaluation Completed")
print("=" * 60)

Teacher Evaluation Started

Evaluating Trajectory 1...
Score: 2.0/10

Evaluating Trajectory 2...
Score: 2.0/10

Evaluating Trajectory 3...
Score: 5.0/10

Evaluating Trajectory 4...
Score: 2.0/10

Evaluating Trajectory 5...
Score: 2.0/10

Teacher Evaluation Completed
